## Configuration & Ngrok Setup

To expose your Ollama server publicly from Kaggle or Colab, you need a free Ngrok Authtoken:
1. Create a free account or log in at [Ngrok Dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
2. Copy your **Your Authtoken** string.
3. Paste it into the `NGROK_AUTH_TOKEN` variable in the cell below.

In [ ]:
# Global Configuration Constants
NGROK_AUTH_TOKEN = "INSERT-YOUR-TOKEN-HERE"
# Insert your Ngrok token above here, within speech marks

## Install Ollama & Pyngrok

In [ ]:
# Ignore non-critical repo update errors (e.g. CRAN mirror sync issues) and install zstd
!apt-get update || true
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install Python dependencies
!pip install pyngrok nest-asyncio requests

## Start Ollama Server & Tunnel

In [ ]:
import os
import subprocess
import time

# Enable cross-origin requests
os.environ['OLLAMA_ORIGINS'] = '*'

# Enable deep logging to see incoming payloads, prompts, and generation metrics
os.environ['OLLAMA_DEBUG'] = '1'
os.environ['OLLAMA_DEBUG_LOG_REQUESTS'] = '1'

# Start Ollama service in background and route all output to ollama.log
print("Starting Ollama server with debug logs enabled...")
log_file = open("ollama.log", "w")
subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=subprocess.STDOUT)

# Give the server a few seconds to fully boot up
time.sleep(3)
print("Server is running!")

## Pull Ollama Model

In [ ]:
!ollama pull Hudson/llama3.1-uncensored:8b

import requests

OLLAMA_URL = "http://localhost:11434/api/generate"

models = ["Hudson/llama3.1-uncensored:8b"]

for model in models:
    print(f"Pre-warming model: {model}...")
    try:
        res = requests.post(OLLAMA_URL, json={"model":model, "keep_alive": -1})
        if res.status_code == 200:
            print(f"Successfully loaded {model} into VRAM.")
    except Exception as e:
        print(f"Failed to pre-warm {model}: {e}")

from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Expose Ollama's default port AND rewrite the host header to bypass the 403 error
tunnel = ngrok.connect(
    11434,
    host_header="localhost:11434"
)
public_url = tunnel.public_url

!curl -H "ngrok-skip-browser-warning: true" {public_url}/v1/models

print(f"\n=======================================================")
print(f"  🔗 YOUR SERVER URL :  ")
print(f"  {public_url}")
print(f"=======================================================\n")

Everything above should be ready in about 5 minutes of total running time.

## Server Status & Keep-Alive Loop

In [ ]:
import time

print("🟢 Ollama server is running and tunnel is active.")
print("Keep this cell running to keep the notebook session active.\n")

uptime_minutes = 0
try:
    while True:
        time.sleep(60)
        uptime_minutes += 1
        print(f"[STATUS] Server running continuously. Uptime: {uptime_minutes} min(s)")
except KeyboardInterrupt:
    print("\n🛑 Server status loop stopped manually.")

🟢 Ollama server is running and tunnel is active.
Keep this cell running to keep the notebook session active.

[STATUS] Server running continuously. Uptime: 1 min(s)
